# Validate the silver tables

A separate pipeline, the job `validate_silver`. It **only reads** the bronze and silver tables. It checks that the tables are consistent, reports how much work is left, and **fails** if a rule that must always hold is broken. Run it after the ingest job, or on a schedule.

It is kept apart from the ingest job on purpose: a validation failure should not mark an ingest run as failed or block it, and the checks can be run on their own at any time, for any schema (the `schema` widget).

| Check | Kind | Meaning |
| --- | --- | --- |
| Every bronze PDF has a record | **must hold** | Each PDF is in `silver_parsed` or in `silver_log`. A PDF with no record would be skipped silently |
| PDFs fully parsed | information | PDFs whose parts are all in `silver_parsed`: these are the ones gold chunks |
| Open failures | information | `silver_failures`: the parts not parsed, and whether a retry makes sense |

"Must hold" checks are about the integrity of the tables, so a failure means a bug or a damaged table. "Information" checks show progress and normal leftovers, and never fail the job.

In [ ]:
import sys
sys.path.insert(0, "../common")  # the shared table names
from names import Names

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("bronze_schema"), dbutils.widgets.get("silver_schema"))


def scalar(sql):
    return spark.sql(sql).first()[0] or 0


# (name, number found, must be zero?)
checks = [
    ("PDFs in bronze with no parts in silver_parts", scalar(f"""
        SELECT count(*) FROM {n.bronze} WHERE path NOT IN (SELECT path FROM {n.parts})"""), True),
    ("parts with no record in silver_parsed or silver_log", scalar(f"""
        SELECT count(*) FROM {n.parts} t
        WHERE NOT EXISTS (SELECT 1 FROM {n.parsed} p WHERE p.path = t.path AND p.part_no = t.part_no)
          AND NOT EXISTS (SELECT 1 FROM {n.log} l WHERE l.stage = 'parse' AND l.path = t.path AND l.part_no = t.part_no)"""), True),
]
info = {
    "PDFs in bronze": scalar(f"SELECT count(*) FROM {n.bronze}"),
    "parts (silver_parts)": scalar(f"SELECT count(*) FROM {n.parts}"),
    "parts parsed (silver_parsed)": scalar(f"SELECT count(*) FROM {n.parsed}"),
    "PDFs fully parsed (all parts in silver_parsed)": scalar(f"""
        SELECT count(*) FROM (SELECT path, count(*) AS n FROM {n.parts} GROUP BY path) t
        JOIN (SELECT path, count(*) AS n FROM {n.parsed} GROUP BY path) p ON p.path = t.path AND p.n = t.n"""),
    "parts waiting for a parse retry": scalar(f"SELECT count(*) FROM {n.failures} WHERE stage = 'parse'"),
}

print("MUST HOLD")
for name, found, _ in checks:
    print(f"  {'OK  ' if found == 0 else 'FAIL'} {name}: {found}")
print("\nINFORMATION")
for name, value in info.items():
    print(f"  {value:>8}  {name}")

## Open failures

What is still open in `silver_failures`: the parts not parsed. A retryable failure is picked up by the next ingest run. A failure that is not retryable (a failed conversion in the parser) needs a different fix.

In [ ]:
display(spark.table(n.failures).groupBy("retryable").count().orderBy("retryable"))
display(spark.table(n.failures).orderBy("path", "part_no").limit(100))

## Result

Fails the task if a "must hold" check found a problem, so the job run shows red.

In [ ]:
broken = [f"{name}: {found}" for name, found, must_be_zero in checks if must_be_zero and found != 0]
assert not broken, "Validation failed:\n  " + "\n  ".join(broken)
print("All checks that must hold passed.")